# Automatización y Control de Recobros Legales

## Objetivo del ejercicio

El objetivo de este análisis es diseñar una herramienta automatizada que permita calcular, controlar y monitorear los recobros asociados a los servicios legales prestados a los fondos y compañías del portafolio.

La solución busca:

- Minimizar errores manuales en el cálculo de recobros.
- Estandarizar el proceso mensual de asignación de costos legales.
- Facilitar la coordinación entre el equipo legal y contable.
- Mejorar la oportunidad y trazabilidad de los cobros.
- Generar controles operativos y financieros sobre la actividad legal recobrable.

La metodología desarrollada utiliza horas recobrables reportadas por empleado, costos mensuales y supuestos de jornada laboral para construir un motor automatizado de cálculo de valores recobrables.
```


In [1]:
import pandas as pd
import numpy as np


# LOAD RECOVERY DATA


legal = pd.read_csv(
    "../data/Recobros CSV.csv",
    sep=";",
    encoding="latin1"
)

legal.columns = (
    legal.columns
    .str.strip()
)

legal.head()

,ï»¿Fondo,Empresa,Mes,Empleado,Horas Dedicadas,Horas Recobrables
0,Fondo III,E2,12,Juan,4.5,1.8
1,Fondo III,E2,1,Lucas,6.0,2.4
2,Fondo III,E2,1,Laura,49.5,19.8
3,Fondo I,E2,8,Laura,3.5,1.4
4,Fondo IV,E2,7,Juan,5.0,2.0


In [2]:
#Limpiar base de datos
legal.columns = (
    legal.columns
    .str.replace("ï»¿", "", regex=False)
    .str.strip()
)

legal.head()

,Fondo,Empresa,Mes,Empleado,Horas Dedicadas,Horas Recobrables
0,Fondo III,E2,12,Juan,4.5,1.8
1,Fondo III,E2,1,Lucas,6.0,2.4
2,Fondo III,E2,1,Laura,49.5,19.8
3,Fondo I,E2,8,Laura,3.5,1.4
4,Fondo IV,E2,7,Juan,5.0,2.0


In [3]:
legal.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 56 entries, 0 to 55
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Fondo              56 non-null     object 
 1   Empresa            56 non-null     object 
 2   Mes                56 non-null     int64  
 3   Empleado           56 non-null     object 
 4   Horas Dedicadas    56 non-null     float64
 5   Horas Recobrables  56 non-null     float64
dtypes: float64(2), int64(1), object(3)
memory usage: 2.8+ KB


In [6]:
employee_costs = {
    "Juan": 11145783.22,
    "Lucas": 4525977.66,
    "Laura": 5768747.42
}

weekly_hours = 42

monthly_hours = (
    weekly_hours * 52 / 12
)

hourly_rates = {
    employee: cost / monthly_hours
    for employee, cost in employee_costs.items()
}

hourly_rates

{'Juan': 61240.567142857144,
 'Lucas': 24868.00912087912,
 'Laura': 31696.414395604395}

## Aplicación de tarifas horarias

Una vez calculadas las horas laborales mensuales estimadas, se procede a calcular la tarifa horaria de cada empleado del equipo legal.

Estas tarifas permiten distribuir automáticamente el costo de los servicios legales según las horas recobrables trabajadas para cada fondo o compañía del portafolio.

El objetivo de este paso es automatizar el proceso de asignación de costos y minimizar errores manuales en el cálculo mensual de recobros.


In [7]:
legal["Hourly_Rate"] = (
    legal["Empleado"]
    .map(hourly_rates)
)

legal.head()

,Fondo,Empresa,Mes,Empleado,Horas Dedicadas,Horas Recobrables,Hourly_Rate
0,Fondo III,E2,12,Juan,4.5,1.8,61240.567143
1,Fondo III,E2,1,Lucas,6.0,2.4,24868.009121
2,Fondo III,E2,1,Laura,49.5,19.8,31696.414396
3,Fondo I,E2,8,Laura,3.5,1.4,31696.414396
4,Fondo IV,E2,7,Juan,5.0,2.0,61240.567143


## Cálculo del valor recobrable

El valor recobrable corresponde al costo asociado a las horas recobrables trabajadas por cada empleado.

Este cálculo permite estimar automáticamente el monto que debe ser cobrado a cada fondo o compañía por los servicios legales prestados.

In [8]:
legal["Recovery_Amount"] = (
    legal["Horas Recobrables"]
    *
    legal["Hourly_Rate"]
)

legal.head()

,Fondo,Empresa,Mes,Empleado,Horas Dedicadas,Horas Recobrables,Hourly_Rate,Recovery_Amount
0,Fondo III,E2,12,Juan,4.5,1.8,61240.567143,110233.020857
1,Fondo III,E2,1,Lucas,6.0,2.4,24868.009121,59683.221890
2,Fondo III,E2,1,Laura,49.5,19.8,31696.414396,627589.005033
3,Fondo I,E2,8,Laura,3.5,1.4,31696.414396,44374.980154
4,Fondo IV,E2,7,Juan,5.0,2.0,61240.567143,122481.134286


## Resumen consolidado de recobros por fondo

Con el objetivo de facilitar el proceso de facturación y seguimiento financiero, se consolida el valor total recobrable por fondo.

Este análisis permite identificar qué fondos concentran mayores valores de recobro y sirve como insumo para el proceso mensual de cobro y conciliación con contabilidad.


In [9]:
fund_summary = legal.groupby(
    "Fondo",
    as_index=False
)["Recovery_Amount"].sum()

fund_summary = fund_summary.sort_values(
    "Recovery_Amount",
    ascending=False
)

fund_summary


,Fondo,Recovery_Amount
2,Fondo III,9.191329e+06
1,Fondo II,2.837663e+06
0,Fondo I,1.806230e+06
3,Fondo IV,1.766183e+06


## Control mensual de recobros

Se consolida el valor recobrable total por mes con el objetivo de monitorear la evolución mensual de los servicios legales prestados a los fondos y compañías del portafolio.

Este control facilita la identificación oportuna de variaciones, retrasos o concentraciones inusuales de recobros.

In [10]:
monthly_summary = legal.groupby(
    "Mes",
    as_index=False
)["Recovery_Amount"].sum()

monthly_summary

,Mes,Recovery_Amount
0,1,6.872722e+05
1,2,9.134084e+05
2,3,1.525498e+06
3,4,7.957763e+04
4,5,1.118652e+06
5,6,2.592664e+06
6,7,5.532078e+06
7,8,3.530041e+05
8,9,1.275291e+06
9,10,1.044456e+06


## Compañías con mayores valores recobrables

Se identifican las compañías del portafolio que presentan mayores valores recobrables asociados a servicios legales.

Este análisis permite priorizar validaciones operativas y realizar seguimiento focalizado sobre los principales drivers de recobro.


In [11]:
company_summary = legal.groupby(
    "Empresa",
    as_index=False
)["Recovery_Amount"].sum()

company_summary = company_summary.sort_values(
    "Recovery_Amount",
    ascending=False
)

company_summary

,Empresa,Recovery_Amount
3,E3,8.291906e+06
5,E5,2.588149e+06
2,E2,1.690229e+06
6,E6,7.946372e+05
8,E8,6.217002e+05
0,E1,5.162248e+05
4,E4,4.206074e+05
7,E7,3.674434e+05
1,E10,1.837217e+05
9,E9,1.267857e+05


## Visualización de recobros por fondo

Se construye una visualización consolidada de los valores recobrables por fondo con el objetivo de facilitar el análisis ejecutivo y la priorización de cobros.

La gráfica permite identificar rápidamente cuáles fondos concentran la mayor proporción de servicios legales recobrables.


In [12]:
import plotly.express as px

fig = px.bar(
    fund_summary,
    x="Fondo",
    y="Recovery_Amount",
    color="Fondo",
    title="Valor Recobrable por Fondo"
)

fig.update_layout(
    template="plotly_white",
    yaxis_title="Valor Recobrable (COP)",
    xaxis_title="Fondo"
)

fig.show()

## Evolución mensual de recobros

Se analiza la evolución mensual de los valores recobrables con el fin de monitorear tendencias, concentraciones de trabajo legal y posibles variaciones en la actividad operativa.

Este análisis facilita la planeación financiera y la validación oportuna de cobros.


In [13]:
fig = px.line(
    monthly_summary,
    x="Mes",
    y="Recovery_Amount",
    markers=True,
    title="Evolución Mensual de Recobros"
)

fig.update_layout(
    template="plotly_white",
    yaxis_title="Valor Recobrable (COP)",
    xaxis_title="Mes"
)

fig.show()

## Compañías con mayor valor recobrable

Se identifican las compañías del portafolio que generan mayores valores recobrables asociados a servicios legales.

Este análisis permite focalizar seguimiento operativo y validar oportunamente los principales componentes del proceso de recobro.


In [14]:
top_companies = (
    company_summary
    .head(10)
)

fig = px.bar(
    top_companies,
    x="Recovery_Amount",
    y="Empresa",
    orientation="h",
    title="Top Compañías por Valor Recobrable"
)

fig.update_layout(
    template="plotly_white",
    yaxis_title="Empresa",
    xaxis_title="Valor Recobrable (COP)"
)

fig.show()

# #Conclusiones
El análisis evidencia que el Fondo III concentra la mayor proporción de servicios legales recobrables, representando el principal driver operativo y financiero del proceso de recobros.

La evolución mensual de recobros presenta alta volatilidad operativa, destacándose un pico significativo en el mes 7. Esto evidencia la necesidad de contar con controles automatizados que permitan monitorear oportunamente concentraciones inusuales de actividad legal y facilitar la coordinación con contabilidad.

El análisis evidencia una alta concentración de servicios legales recobrables en pocas compañías del portafolio, especialmente en E3, que representa el principal driver de consumo de recursos legales. Este comportamiento sugiere oportunidades para fortalecer el monitoreo operativo y la priorización de seguimiento sobre compañías de mayor complejidad legal.

